# 01 — Análisis del dataset UrBAN

**Proyecto:** Prototipo de plataforma tecnológica híbrida para el procesamiento digital y visualización de señales bioacústicas de colmenas de abejas melíferas asociadas a *Varroa destructor*.

## Propósito

Este notebook realiza una **auditoría exploratoria y reproducible** del dataset UrBAN antes de construir el `dataset.csv` maestro y extraer características acústicas.

El análisis:

1. Localiza los archivos de anotaciones de UrBAN.
2. Carga y normaliza las inspecciones de 2021 y 2022.
3. Identifica las observaciones de categoría `varroa`.
4. Conserva el valor numérico original de `Action detail`.
5. Analiza fechas, colmenas y distribución de mediciones.
6. Inspecciona los nombres de archivos de audio disponibles localmente.
7. Verifica la posibilidad de asociar audio con colmena y fecha/hora.
8. Estima la cantidad potencial de segmentos de 2 segundos.
9. Genera alertas de calidad y problemas de etiquetado.
10. Exporta tablas auxiliares para el siguiente notebook.

> **Importante:** este notebook **no descarga automáticamente miles de horas de audio**. El procesamiento de audio se hará únicamente sobre los archivos que el investigador haya descargado y colocado en `training/data/raw/urban/`.

### Criterios metodológicos

- Segmentos objetivo: **2 segundos**, de acuerdo con el proyecto aprobado.
- Etiqueta binaria derivada: `0` si la medición de Varroa es `< 3`; `1` si es `>= 3`.
- El valor original se conserva en `varroa_measurement`.
- La etiqueta se deriva; **no se presenta como una etiqueta original de UrBAN**.
- La separación train/validation/test debe hacerse por **colmena (`hive_id`)**, no por segmento, para reducir fuga de información.
- Una medición de Varroa no debe utilizarse para etiquetar indiscriminadamente todo el audio de un período sin una ventana temporal explícitamente definida.

In [ ]:
from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Ruta base del repositorio.
# Si ejecutas el notebook desde training/notebooks/, esta ruta es correcta.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == "notebooks" else Path.cwd()

# Si el cálculo anterior no coincide con tu ubicación, cambia manualmente esta ruta.
DATA_ROOT = REPO_ROOT / "training" / "data"
RAW_DIR = DATA_ROOT / "raw" / "urban"
METADATA_DIR = DATA_ROOT / "metadata"
PROCESSED_DIR = DATA_ROOT / "processed"
SPLITS_DIR = DATA_ROOT / "splits"

for p in [RAW_DIR, METADATA_DIR, PROCESSED_DIR, SPLITS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Directorio de trabajo:", Path.cwd())
print("Repositorio:", REPO_ROOT)
print("Datos:", DATA_ROOT)
print("Audio:", RAW_DIR)

In [ ]:
# Dependencias esperadas
# Descomenta la línea si falta alguna en tu entorno:
# %pip install pandas numpy matplotlib openpyxl soundfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Configuración de parámetros

Los parámetros de esta sección quedan centralizados para que el análisis sea reproducible.

La asociación temporal audio → medición de Varroa es deliberadamente configurable. **No se debe asumir que una medición de un día etiqueta automáticamente todo el mes.**

Para una primera construcción conservadora se utiliza una ventana de ±24 horas alrededor de la medición. Este valor debe validarse antes de la evaluación final del modelo.

In [ ]:
# -----------------------------
# Parámetros del experimento
# -----------------------------
TARGET_SAMPLE_RATE = 16_000
SEGMENT_DURATION_SEC = 2.0

# Umbral binario derivado utilizado como punto de partida.
# 0 = medición < 3
# 1 = medición >= 3
VARROA_THRESHOLD = 3.0

# Ventana temporal de asociación audio -> inspección Varroa.
# Se recomienda validar este supuesto metodológicamente.
LABEL_WINDOW_HOURS = 24

# Extensiones de audio que se reconocerán.
AUDIO_EXTENSIONS = {".wav", ".wave", ".mp3", ".flac", ".ogg", ".m4a", ".webm"}

# Guardar resultados auxiliares
EXPORT_AUXILIARY = True

## 2. Localización de las anotaciones

UrBAN contiene anotaciones de inspecciones. El análisis espera encontrar:

- `inspections_2021.csv`
- `inspections_2022.csv`

en `training/data/metadata/`.

Si todavía no están allí, también se intenta buscarlas dentro de `training/data/raw/urban/`.

In [ ]:
def find_file(filename, roots):
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob(filename))
    return candidates[0] if candidates else None

inspection_2021_path = find_file(
    "inspections_2021.csv",
    [METADATA_DIR, RAW_DIR, DATA_ROOT]
)
inspection_2022_path = find_file(
    "inspections_2022.csv",
    [METADATA_DIR, RAW_DIR, DATA_ROOT]
)

print("inspections_2021.csv:", inspection_2021_path)
print("inspections_2022.csv:", inspection_2022_path)

if inspection_2021_path is None and inspection_2022_path is None:
    raise FileNotFoundError(
        "No se encontraron inspections_2021.csv ni inspections_2022.csv. "
        "Colócalos en training/data/metadata/."
    )

## 3. Carga y normalización de inspecciones

In [ ]:
def load_inspection(path, year):
    if path is None:
        return pd.DataFrame()

    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    df["source_file"] = path.name
    df["source_year"] = year
    return df

inspections_2021 = load_inspection(inspection_2021_path, 2021)
inspections_2022 = load_inspection(inspection_2022_path, 2022)

print("2021:", inspections_2021.shape)
print("2022:", inspections_2022.shape)

if not inspections_2021.empty:
    display(inspections_2021.head())
if not inspections_2022.empty:
    display(inspections_2022.head())

In [ ]:
# Comprobación de columnas esperadas por UrBAN
EXPECTED_COLUMNS = [
    "Date", "Tag number", "Category", "Action detail",
    "Queen status", "Is alive", "Report notes"
]

for year, df in [(2021, inspections_2021), (2022, inspections_2022)]:
    missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    print(f"{year}: columnas faltantes -> {missing}")
    print("Columnas disponibles:", list(df.columns))
    print()

## 4. Estandarización de campos

Se mantienen los nombres originales para trazabilidad y se crean campos normalizados:

- `date`
- `hive_id`
- `category`
- `action_detail`
- `varroa_measurement`

La conversión numérica de `Action detail` se hace de forma segura. Los valores que no puedan convertirse se conservan para inspección.

In [ ]:
def normalize_inspections(df):
    if df.empty:
        return df.copy()

    out = df.copy()
    out["date"] = pd.to_datetime(out["Date"], errors="coerce", dayfirst=False)
    out["hive_id"] = out["Tag number"].astype(str).str.strip()
    out["category"] = out["Category"].astype(str).str.strip().str.lower()
    out["action_detail"] = out["Action detail"].astype(str).str.strip()

    # Convierte números; admite coma decimal como caso defensivo.
    cleaned = (
        out["action_detail"]
        .str.replace(",", ".", regex=False)
        .str.extract(r"([-+]?\d+(?:\.\d+)?)", expand=False)
    )
    out["action_detail_numeric"] = pd.to_numeric(cleaned, errors="coerce")

    # Solo para observaciones de Varroa.
    out["varroa_measurement"] = np.where(
        out["category"].eq("varroa"),
        out["action_detail_numeric"],
        np.nan
    )

    return out

inspections = pd.concat(
    [
        normalize_inspections(inspections_2021),
        normalize_inspections(inspections_2022)
    ],
    ignore_index=True
)

print("Total de registros:", len(inspections))
display(inspections.head())

## 5. Inventario de categorías

Antes de filtrar Varroa se revisan las categorías presentes en las inspecciones. Esto evita asumir que el dataset contiene únicamente observaciones de la enfermedad.

In [ ]:
category_counts = (
    inspections["category"]
    .value_counts(dropna=False)
    .rename_axis("category")
    .reset_index(name="records")
)

display(category_counts)

In [ ]:
plt.figure(figsize=(10, 5))
category_counts.head(20).plot(
    kind="bar",
    x="category",
    y="records",
    legend=False,
    figsize=(10, 5)
)
plt.title("Registros por categoría de inspección")
plt.xlabel("Categoría")
plt.ylabel("Cantidad de registros")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Registros asociados a Varroa

Aquí se construye la tabla de referencia que posteriormente se utilizará para el etiquetado.

El valor `varroa_measurement` se conserva sin modificar conceptualmente: es la medición original disponible en `Action detail`.

In [ ]:
varroa = inspections.loc[
    inspections["category"].eq("varroa")
].copy()

varroa["varroa_date"] = varroa["date"]
varroa["hive_id"] = varroa["hive_id"].astype(str)

# Etiqueta binaria derivada, NO original de UrBAN.
varroa["varroa_label"] = np.where(
    varroa["varroa_measurement"].notna(),
    (varroa["varroa_measurement"] >= VARROA_THRESHOLD).astype("Int64"),
    pd.NA
)

print("Registros Varroa:", len(varroa))
print("Colmenas con Varroa:", varroa["hive_id"].nunique())
display(
    varroa[
        [
            "varroa_date", "hive_id",
            "varroa_measurement", "varroa_label",
            "source_file", "action_detail"
        ]
    ].sort_values(["varroa_date", "hive_id"]).head(100)
)

## 7. Calidad de las mediciones de Varroa

In [ ]:
print("Valores Varroa no numéricos:")
display(
    varroa.loc[
        varroa["varroa_measurement"].isna(),
        ["varroa_date", "hive_id", "action_detail", "source_file"]
    ].head(50)
)

print("Valores numéricos:")
display(varroa["varroa_measurement"].describe())

print("Duplicados por fecha + colmena:")
duplicates = (
    varroa.groupby(["varroa_date", "hive_id"], dropna=False)
    .size()
    .reset_index(name="count")
)
display(duplicates.query("count > 1"))

In [ ]:
# Distribución de las mediciones
numeric_varroa = varroa["varroa_measurement"].dropna()

if not numeric_varroa.empty:
    plt.figure(figsize=(9, 5))
    plt.hist(numeric_varroa, bins=min(20, max(5, numeric_varroa.nunique())))
    plt.axvline(VARROA_THRESHOLD, linestyle="--", label=f"Umbral = {VARROA_THRESHOLD}")
    plt.title("Distribución de mediciones de Varroa")
    plt.xlabel("varroa_measurement")
    plt.ylabel("Número de observaciones")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 8. Distribución temporal

La fecha de inspección es crítica para relacionar correctamente las grabaciones de audio con la medición de Varroa.

In [ ]:
varroa_by_date = (
    varroa.groupby("varroa_date", dropna=False)
    .agg(
        records=("hive_id", "size"),
        hives=("hive_id", "nunique"),
        mean_measurement=("varroa_measurement", "mean")
    )
    .reset_index()
    .sort_values("varroa_date")
)

display(varroa_by_date)

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(
    varroa_by_date["varroa_date"],
    varroa_by_date["mean_measurement"],
    marker="o"
)
plt.axhline(VARROA_THRESHOLD, linestyle="--", label=f"Umbral = {VARROA_THRESHOLD}")
plt.title("Promedio de medición de Varroa por fecha")
plt.xlabel("Fecha")
plt.ylabel("Promedio de varroa_measurement")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Distribución por colmena

Para la evaluación posterior se necesita conocer cuántas observaciones tiene cada colmena. Esto será importante al construir los grupos de train/validation/test.

In [ ]:
hive_summary = (
    varroa.groupby("hive_id", dropna=False)
    .agg(
        varroa_records=("varroa_measurement", "size"),
        valid_measurements=("varroa_measurement", lambda s: s.notna().sum()),
        min_varroa=("varroa_measurement", "min"),
        max_varroa=("varroa_measurement", "max"),
        mean_varroa=("varroa_measurement", "mean"),
        positive_labels=("varroa_label", lambda s: (s == 1).sum()),
        negative_labels=("varroa_label", lambda s: (s == 0).sum())
    )
    .reset_index()
    .sort_values("hive_id")
)

display(hive_summary)

## 10. Inventario de audio local

El notebook no descarga el dataset completo. En su lugar, inspecciona los archivos que ya estén disponibles localmente.

UrBAN documenta nombres de audio que contienen fecha, hora y número de colmena. El parser siguiente es deliberadamente flexible y debe validarse contra los nombres reales descargados.

In [ ]:
audio_files = []
for ext in AUDIO_EXTENSIONS:
    audio_files.extend(RAW_DIR.rglob(f"*{ext}"))

audio_files = sorted(set(audio_files))

print(f"Archivos de audio encontrados: {len(audio_files):,}")

if audio_files:
    audio_inventory = pd.DataFrame({
        "audio_path": [str(p) for p in audio_files],
        "filename": [p.name for p in audio_files],
        "extension": [p.suffix.lower() for p in audio_files],
        "size_mb": [p.stat().st_size / (1024**2) for p in audio_files]
    })
    display(audio_inventory.head(20))
    print(f"Tamaño total local: {audio_inventory['size_mb'].sum()/1024:.2f} GB")
else:
    audio_inventory = pd.DataFrame(
        columns=["audio_path", "filename", "extension", "size_mb"]
    )
    print("No hay audio local todavía.")

### Parser de nombres de audio

Se intenta interpretar un patrón del tipo:

`DD-MM-YYYY_HHhMM_HIVE_Tag.wav`

Si el patrón real de los archivos descargados difiere, **no se debe forzar la asociación**. El notebook marcará esos archivos como no interpretados para revisión.

In [ ]:
def parse_urban_filename(filename):
    name = Path(filename).stem

    # Patrón esperado: DD-MM-YYYY_HHhMM_HIVE_Tag
    patterns = [
        re.compile(
            r"^(?P<day>\d{2})-(?P<month>\d{2})-(?P<year>\d{4})_"
            r"(?P<hour>\d{1,2})h(?P<minute>\d{2})_"
            r"(?P<hive>[^_]+)"
            r"(?:_.*)?$",
            re.IGNORECASE
        ),
        re.compile(
            r"^(?P<day>\d{2})-(?P<month>\d{2})-(?P<year>\d{4})_"
            r"(?P<hour>\d{1,2})h(?P<minute>\d{2})_"
            r"(?P<hive>\d+).*?$",
            re.IGNORECASE
        )
    ]

    for pattern in patterns:
        m = pattern.match(name)
        if m:
            try:
                dt = pd.Timestamp(
                    year=int(m.group("year")),
                    month=int(m.group("month")),
                    day=int(m.group("day")),
                    hour=int(m.group("hour")),
                    minute=int(m.group("minute"))
                )
                return {
                    "audio_date": dt.date(),
                    "audio_start": dt,
                    "hive_id": str(m.group("hive")).strip(),
                    "parse_status": "PARSED"
                }
            except ValueError:
                pass

    return {
        "audio_date": pd.NaT,
        "audio_start": pd.NaT,
        "hive_id": None,
        "parse_status": "UNPARSED"
    }

if not audio_inventory.empty:
    parsed = audio_inventory["filename"].apply(parse_urban_filename).apply(pd.Series)
    audio_inventory = pd.concat([audio_inventory, parsed], axis=1)
    display(audio_inventory.head(20))
    print(audio_inventory["parse_status"].value_counts(dropna=False))
else:
    print("Se omite parsing porque no hay archivos de audio.")

## 11. Validación del parser y cobertura por colmena

Se compara el conjunto de colmenas detectadas en los nombres de audio con las colmenas que tienen mediciones de Varroa.

In [ ]:
if not audio_inventory.empty:
    audio_hives = set(
        audio_inventory.loc[
            audio_inventory["parse_status"].eq("PARSED"),
            "hive_id"
        ].dropna().astype(str)
    )
else:
    audio_hives = set()

varroa_hives = set(varroa["hive_id"].dropna().astype(str))

print("Colmenas con medición Varroa:", len(varroa_hives))
print("Colmenas detectadas en audio:", len(audio_hives))
print("Intersección:", len(varroa_hives & audio_hives))
print("Varroa sin audio identificado:", sorted(varroa_hives - audio_hives)[:50])
print("Audio sin medición Varroa:", sorted(audio_hives - varroa_hives)[:50])

## 12. Resumen de fechas de audio

La relación correcta requiere comparar la fecha/hora del audio con la fecha de inspección de Varroa, no solamente el número de colmena.

In [ ]:
if not audio_inventory.empty:
    audio_inventory["audio_date_only"] = pd.to_datetime(
        audio_inventory["audio_start"], errors="coerce"
    ).dt.normalize()

    audio_date_summary = (
        audio_inventory.loc[audio_inventory["parse_status"].eq("PARSED")]
        .groupby("audio_date_only")
        .agg(
            audio_files=("filename", "size"),
            hives=("hive_id", "nunique")
        )
        .reset_index()
        .sort_values("audio_date_only")
    )

    display(audio_date_summary.head(50))
else:
    print("No hay audio para resumir.")

## 13. Asociación preliminar audio → medición de Varroa

Se crea una asociación únicamente cuando:

- el audio tiene `hive_id` interpretable;
- existe una medición numérica de Varroa para la misma colmena;
- la diferencia temporal está dentro de `LABEL_WINDOW_HOURS`.

La asociación queda marcada con `label_status`.

**No se genera todavía el `dataset.csv` definitivo**: esta etapa sirve para conocer la cobertura y detectar ambigüedades.

In [ ]:
def build_audio_label_candidates(audio_df, varroa_df, window_hours):
    if audio_df.empty or varroa_df.empty:
        return pd.DataFrame()

    a = audio_df.loc[
        audio_df["parse_status"].eq("PARSED")
        & audio_df["audio_start"].notna()
        & audio_df["hive_id"].notna()
    ].copy()

    v = varroa_df.loc[
        varroa_df["varroa_measurement"].notna()
        & varroa_df["varroa_date"].notna()
    ].copy()

    if a.empty or v.empty:
        return pd.DataFrame()

    a["hive_id"] = a["hive_id"].astype(str)
    v["hive_id"] = v["hive_id"].astype(str)

    # Para cada audio, encontrar todas las mediciones posibles de la misma colmena.
    candidates = a.merge(
        v[
            [
                "hive_id", "varroa_date",
                "varroa_measurement", "varroa_label",
                "source_file"
            ]
        ],
        on="hive_id",
        how="left",
        suffixes=("", "_varroa")
    )

    candidates["time_delta_minutes"] = (
        candidates["audio_start"] - candidates["varroa_date"]
    ).abs().dt.total_seconds() / 60

    candidates["within_window"] = (
        candidates["time_delta_minutes"] <= window_hours * 60
    )

    return candidates

if not audio_inventory.empty:
    candidates = build_audio_label_candidates(
        audio_inventory,
        varroa,
        LABEL_WINDOW_HOURS
    )
else:
    candidates = pd.DataFrame()

print("Candidatos de asociación:", len(candidates))
if not candidates.empty:
    display(
        candidates[
            [
                "filename", "hive_id", "audio_start",
                "varroa_date", "varroa_measurement",
                "varroa_label", "time_delta_minutes",
                "within_window"
            ]
        ].head(50)
    )

## 14. Resolución de asociaciones ambiguas

Para cada archivo de audio se selecciona la medición de Varroa temporalmente más cercana **solo si está dentro de la ventana configurada**.

Si hay dos mediciones igualmente cercanas, el registro queda `AMBIGUOUS` y no debe utilizarse automáticamente para entrenamiento.

In [ ]:
def resolve_labels(candidates):
    if candidates.empty:
        return pd.DataFrame()

    c = candidates.copy()

    valid = c[c["within_window"]].copy()
    if valid.empty:
        return pd.DataFrame(columns=list(c.columns) + ["label_status"])

    # Orden temporal por cercanía.
    valid = valid.sort_values(
        ["audio_path", "time_delta_minutes", "varroa_date"]
    )

    rows = []
    for audio_path, group in valid.groupby("audio_path", sort=False):
        min_delta = group["time_delta_minutes"].min()
        nearest = group[group["time_delta_minutes"].eq(min_delta)]

        row = nearest.iloc[0].copy()

        if len(nearest) > 1:
            row["label_status"] = "AMBIGUOUS"
        else:
            row["label_status"] = "VALID"

        rows.append(row)

    return pd.DataFrame(rows)

resolved = resolve_labels(candidates)

print("Asociaciones resueltas:", len(resolved))
if not resolved.empty:
    display(
        resolved[
            [
                "filename", "hive_id", "audio_start",
                "varroa_date", "varroa_measurement",
                "varroa_label", "time_delta_minutes",
                "label_status"
            ]
        ].head(50)
    )

## 15. Detección de audio sin etiqueta y problemas de cobertura

Estos conteos son esenciales para no interpretar el tamaño total del repositorio como tamaño efectivo del dataset de entrenamiento.

In [ ]:
if not audio_inventory.empty:
    total_audio = len(audio_inventory)
    parsed_audio = int((audio_inventory["parse_status"] == "PARSED").sum())
    labeled_audio = int(
        resolved.loc[resolved["label_status"] == "VALID", "audio_path"].nunique()
        if not resolved.empty else 0
    )
    ambiguous_audio = int(
        resolved.loc[resolved["label_status"] == "AMBIGUOUS", "audio_path"].nunique()
        if not resolved.empty else 0
    )

    coverage_summary = pd.DataFrame({
        "metric": [
            "audio_files_total",
            "audio_files_parsed",
            "audio_files_validly_labeled",
            "audio_files_ambiguous"
        ],
        "value": [
            total_audio,
            parsed_audio,
            labeled_audio,
            ambiguous_audio
        ]
    })
    display(coverage_summary)
else:
    print("No hay audio local.")

## 16. Estimación de segmentos de 2 segundos

Si una grabación dura `D` segundos, el número máximo de segmentos no solapados de 2 segundos es:

`floor(D / 2)`

Esta es una **estimación**. La cantidad real dependerá de la duración de cada archivo y de si posteriormente se decide utilizar ventanas solapadas.

No se recomienda crear millones de filas en memoria sin antes comprobar el volumen real.

In [ ]:
# Duración conocida opcionalmente a partir del tamaño/metadata no es suficiente.
# Para calcular duración real se intenta usar soundfile si está instalado.

try:
    import soundfile as sf
    SOUNDFILE_AVAILABLE = True
except ImportError:
    SOUNDFILE_AVAILABLE = False

if SOUNDFILE_AVAILABLE and not audio_inventory.empty:
    durations = []
    sample_rates = []

    for path in audio_inventory["audio_path"]:
        try:
            info = sf.info(path)
            durations.append(float(info.duration))
            sample_rates.append(int(info.samplerate))
        except Exception:
            durations.append(np.nan)
            sample_rates.append(np.nan)

    audio_inventory["duration_sec"] = durations
    audio_inventory["source_sample_rate_hz"] = sample_rates
    audio_inventory["estimated_2s_segments"] = np.floor(
        audio_inventory["duration_sec"] / SEGMENT_DURATION_SEC
    ).astype("Int64")

    display(
        audio_inventory[
            [
                "filename", "hive_id", "audio_start",
                "duration_sec", "source_sample_rate_hz",
                "estimated_2s_segments"
            ]
        ].head(20)
    )

    print(
        "Segmentos estimados:",
        int(audio_inventory["estimated_2s_segments"].sum(skipna=True))
    )
else:
    print(
        "soundfile no está disponible o no hay audio. "
        "La estimación de segmentos se hará en el notebook de construcción."
    )

## 17. Verificación de frecuencia de muestreo

El proyecto/prototipo utilizará una representación normalizada a **16 kHz** para el procesamiento posterior.

El audio original de UrBAN puede tener otra frecuencia de muestreo; por eso se conserva `source_sample_rate_hz` y el procesamiento posterior debe realizar el remuestreo explícitamente.

In [ ]:
if not audio_inventory.empty and "source_sample_rate_hz" in audio_inventory:
    sr_counts = (
        audio_inventory["source_sample_rate_hz"]
        .value_counts(dropna=False)
        .rename_axis("sample_rate_hz")
        .reset_index(name="files")
    )
    display(sr_counts)

## 18. Construcción de un resumen de calidad

Se generan indicadores para decidir si el dataset está listo para pasar al notebook `02_build_dataset.ipynb`.

In [ ]:
quality = []

def add_quality(check, value, status, detail=""):
    quality.append({
        "check": check,
        "value": value,
        "status": status,
        "detail": detail
    })

add_quality(
    "inspections_loaded",
    len(inspections),
    "OK" if len(inspections) > 0 else "ERROR",
    "Registros de inspección cargados."
)

add_quality(
    "varroa_records",
    len(varroa),
    "OK" if len(varroa) > 0 else "ERROR",
    "Registros con Category=varroa."
)

invalid_measurements = int(varroa["varroa_measurement"].isna().sum())
add_quality(
    "varroa_non_numeric_measurements",
    invalid_measurements,
    "OK" if invalid_measurements == 0 else "REVIEW",
    "Valores de Action detail que no pudieron convertirse a número."
)

if not audio_inventory.empty:
    unparsed = int((audio_inventory["parse_status"] != "PARSED").sum())
    add_quality(
        "audio_unparsed",
        unparsed,
        "OK" if unparsed == 0 else "REVIEW",
        "Archivos cuyo nombre no pudo interpretarse."
    )

    valid_labeled = (
        int(resolved.loc[resolved["label_status"] == "VALID", "audio_path"].nunique())
        if not resolved.empty else 0
    )
    add_quality(
        "audio_validly_labeled",
        valid_labeled,
        "OK" if valid_labeled > 0 else "REVIEW",
        f"Con ventana temporal de ±{LABEL_WINDOW_HOURS} h."
    )
else:
    add_quality(
        "audio_available",
        0,
        "REVIEW",
        "No se ha descargado audio local; el análisis de audio queda pendiente."
    )

quality_df = pd.DataFrame(quality)
display(quality_df)

## 19. Resumen de clases derivadas

La distribución de clases debe revisarse **después** de aplicar la regla temporal de etiquetado. Contar únicamente las mediciones de Varroa no representa el balance del dataset de segmentos.

In [ ]:
if not resolved.empty:
    valid_resolved = resolved[resolved["label_status"].eq("VALID")].copy()
    class_summary = (
        valid_resolved["varroa_label"]
        .value_counts(dropna=False)
        .rename_axis("varroa_label")
        .reset_index(name="audio_files")
    )
    class_summary["label_name"] = class_summary["varroa_label"].map({
        0: "No alto",
        1: "Alto"
    })
    display(class_summary)
else:
    print("Todavía no hay asociaciones válidas.")

## 20. Recomendación de split por colmena

Para evitar fuga de información, los segmentos de una misma colmena no deben aparecer simultáneamente en entrenamiento y prueba.

Este notebook solo propone una asignación reproducible de **colmenas**, no crea todavía los segmentos.

In [ ]:
hives_for_split = sorted(varroa_hives)

if len(hives_for_split) >= 3:
    rng = np.random.default_rng(42)
    shuffled = np.array(hives_for_split, dtype=object)
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_test = max(1, round(n * 0.20))
    n_val = max(1, round(n * 0.20))

    # Garantiza que quede al menos una colmena para entrenamiento.
    if n_test + n_val >= n:
        n_val = 1
        n_test = 1

    test_hives = shuffled[:n_test]
    val_hives = shuffled[n_test:n_test+n_val]
    train_hives = shuffled[n_test+n_val:]

    split_hives = pd.DataFrame({
        "hive_id": (
            list(train_hives) +
            list(val_hives) +
            list(test_hives)
        ),
        "split": (
            ["train"] * len(train_hives) +
            ["validation"] * len(val_hives) +
            ["test"] * len(test_hives)
        )
    })

    display(split_hives)
else:
    split_hives = pd.DataFrame(columns=["hive_id", "split"])
    print(
        "No hay suficientes colmenas con mediciones para crear un split "
        "representativo. Se requiere revisar el audio/dataset."
    )

### Regla importante del split

El split anterior es **por colmena**. En el siguiente notebook debe asignarse el mismo `split` a todos los segmentos de esa colmena.

Antes del entrenamiento final se recomienda revisar además que:

- train contenga ambas clases;
- validation contenga ambas clases cuando el número de colmenas lo permita;
- test contenga ambas clases cuando el número de colmenas lo permita.

Con pocas colmenas, una división simple puede producir conjuntos con una sola clase. En ese caso debe diseñarse una estrategia de validación por grupos compatible con el tamaño real del dataset.

## 21. Exportación de resultados auxiliares

Se exportan:

- `varroa_measurements_clean.csv`: mediciones de Varroa normalizadas.
- `audio_inventory.csv`: inventario del audio local.
- `audio_label_candidates.csv`: asociaciones candidatas.
- `audio_label_resolved.csv`: asociaciones resueltas.
- `train_validation_test_hives.csv`: propuesta de split por colmena.

Estos archivos sirven como insumo del siguiente notebook.

In [ ]:
if EXPORT_AUXILIARY:
    varroa_export = varroa[
        [
            "varroa_date", "hive_id",
            "varroa_measurement", "varroa_label",
            "action_detail", "source_file", "source_year"
        ]
    ].sort_values(["varroa_date", "hive_id"])

    varroa_export.to_csv(
        METADATA_DIR / "varroa_measurements_clean.csv",
        index=False
    )

    if not audio_inventory.empty:
        audio_inventory.to_csv(
            METADATA_DIR / "audio_inventory.csv",
            index=False
        )

    if not candidates.empty:
        candidates.to_csv(
            METADATA_DIR / "audio_label_candidates.csv",
            index=False
        )

    if not resolved.empty:
        resolved.to_csv(
            METADATA_DIR / "audio_label_resolved.csv",
            index=False
        )

    if not split_hives.empty:
        split_hives.to_csv(
            SPLITS_DIR / "train_validation_test_hives.csv",
            index=False
        )

    print("Archivos auxiliares exportados en:")
    print(METADATA_DIR)
    print(SPLITS_DIR)

## 22. Conclusiones automáticas

El notebook finaliza con un resumen que debe acompañar la documentación del proyecto.

In [ ]:
print("=" * 70)
print("RESUMEN DEL ANÁLISIS UrBAN")
print("=" * 70)

print(f"Registros de inspección: {len(inspections):,}")
print(f"Registros de Varroa: {len(varroa):,}")
print(f"Colmenas con medición Varroa: {varroa['hive_id'].nunique():,}")

if not varroa["varroa_measurement"].dropna().empty:
    print(
        "Rango de mediciones:",
        f"{varroa['varroa_measurement'].min():.3f}",
        "a",
        f"{varroa['varroa_measurement'].max():.3f}"
    )

print(f"Umbral binario derivado: {VARROA_THRESHOLD}")
print(f"Ventana temporal de asociación: ±{LABEL_WINDOW_HOURS} horas")
print(f"Duración objetivo de segmento: {SEGMENT_DURATION_SEC} segundos")
print(f"Frecuencia normalizada objetivo: {TARGET_SAMPLE_RATE} Hz")

if not audio_inventory.empty:
    print(f"Audio local: {len(audio_inventory):,} archivos")
    print(
        "Audio con nombre interpretable:",
        int((audio_inventory['parse_status'] == 'PARSED').sum())
    )
    if not resolved.empty:
        print(
            "Audio con etiqueta válida:",
            int(
                resolved.loc[
                    resolved["label_status"] == "VALID",
                    "audio_path"
                ].nunique()
            )
        )
else:
    print("Audio local: todavía no disponible")

print("=" * 70)

# Criterios de salida para `02_build_dataset.ipynb`

Antes de continuar, verifica:

- [ ] `inspections_2021.csv` y/o `inspections_2022.csv` fueron cargados.
- [ ] Existen registros `Category == "varroa"`.
- [ ] `varroa_measurement` conserva los valores numéricos originales.
- [ ] El umbral binario está documentado como **derivado**.
- [ ] La ventana temporal de asociación fue revisada.
- [ ] Los nombres de audio reales fueron validados.
- [ ] No se está etiquetando todo el audio de una colmena sin criterio temporal.
- [ ] El split será por `hive_id`.
- [ ] Los WAV originales permanecerán fuera del repositorio Git si su licencia no permite redistribución.
- [ ] La cantidad efectiva de segmentos será calculada a partir del audio realmente seleccionado.

## Próximo notebook

`02_build_dataset.ipynb` debe convertir las asociaciones válidas en el `dataset.csv` maestro con, como mínimo:

`sample_id, audio_file, hive_id, audio_date, audio_start, audio_end, segment_start_sec, segment_end_sec, sample_rate_hz, duration_sec, varroa_date, varroa_measurement, varroa_label, label_time_delta_minutes, label_source, label_status, split, source_dataset`

El siguiente paso será generar los segmentos de 2 segundos y dejar preparada la extracción de MFCC para `03_mfcc_extraction.ipynb`.